### Task 1

In [3]:
import numpy as np
import pandas as pd
import random

attendance_raw = []

cohort_names = ["alpha", "beta", "gamma"]

for i in range(1, 25):
    record = {
        "student_id": f"S{i:03d}",   
        "cohort": random.choice(cohort_names),
        "attended_sessions": random.randint(0, 6),
        "expected_sessions": 6
    }
    attendance_raw.append(record)

attendance = pd.DataFrame(attendance_raw)
attendance.head()
attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   student_id         24 non-null     object
 1   cohort             24 non-null     object
 2   attended_sessions  24 non-null     int64 
 3   expected_sessions  24 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 900.0+ bytes


### Task 2

In [4]:
# Set student_id as index
attendance_indexed = attendance.set_index("student_id")

# Create excused_absences Series
excused_absences = pd.Series(
    [1, 2, 1, 3, 2, 1, 2, 1, 3, 1],
    index=[
        "S001", "S003", "S005", "S010", "S012",
        "S020", "S025", "S030", "S100", "S999"
    ],
    name="excused_absences"
)

# Add excused_absences to attended_sessions
attendance_indexed["adjusted_attendance"] = (
    attendance_indexed["attended_sessions"] +
    excused_absences.reindex(attendance_indexed.index)
)

# Confirm missing values (students without matching IDs)
print(attendance_indexed["adjusted_attendance"].isna())

# Fill missing values with original attended_sessions
attendance_indexed["adjusted_attendance"] = attendance_indexed["adjusted_attendance"].fillna(
    attendance_indexed["attended_sessions"]
)

# Show updated column
attendance_indexed[["attended_sessions", "adjusted_attendance"]]

student_id
S001    False
S002     True
S003    False
S004     True
S005    False
S006     True
S007     True
S008     True
S009     True
S010    False
S011     True
S012    False
S013     True
S014     True
S015     True
S016     True
S017     True
S018     True
S019     True
S020    False
S021     True
S022     True
S023     True
S024     True
Name: adjusted_attendance, dtype: bool


,attended_sessions,adjusted_attendance
student_id,,
S001,5,6.0
S002,4,4.0
S003,5,7.0
S004,3,3.0
S005,2,3.0
S006,1,1.0
S007,6,6.0
S008,4,4.0
S009,5,5.0


### Task 3

In [5]:
attendance.loc[
    attendance["attended_sessions"] % 2 == 0,  
    "cohort"                                   
] = attendance.loc[
    attendance["attended_sessions"] % 2 == 0, "cohort"
].str.capitalize() + "  "       

attendance.loc[:, "cohort"] = attendance["cohort"].str.strip().str.lower()
attendance.loc[:, "cohort"].unique()

array(['alpha', 'beta', 'gamma'], dtype=object)

### Task 4

In [6]:
low_attendance = attendance[attendance["attended_sessions"]<attendance["expected_sessions"]]
avg_attendance_by_cohort = attendance.groupby("cohort")["attended_sessions"].mean()
print(avg_attendance_by_cohort)
print("Cohorts in summary:", avg_attendance_by_cohort.index.tolist())
print("Unique cohorts in data:", attendance["cohort"].unique())

cohort
alpha    3.571429
beta     4.333333
gamma    2.875000
Name: attended_sessions, dtype: float64
Cohorts in summary: ['alpha', 'beta', 'gamma']
Unique cohorts in data: ['alpha' 'beta' 'gamma']


### Task 5

In [7]:
attendance["attendance_ok"] = attendance["attended_sessions"] >= attendance["expected_sessions"]
low_attendance = attendance[attendance["attended_sessions"]<attendance["expected_sessions"]]
print(all(low_attendance["attendance_ok"]==False))

True
